# 1 - Delays

## Importation des modules

In [1]:
# Modules de base
import pandas as pd
import numpy as np
import sys
import warnings
warnings.filterwarnings('ignore')

# Ajout du chemin
sys.path.append('..')

# Modules du package
from tsforecast.delays.data_manager import compare_and_detect_delays, _validate_input_data

## Génération de données synthétiques

### Génération de jeux de données de séries temporelles

In [2]:
# Set random seed for reproducibility
np.random.seed(42)

# Create monthly dates from 2020 to 2024
monthly_dates = pd.date_range('2020-01-01', '2024-01-01', freq='MS')
quarterly_dates = pd.date_range('2020-01-01', '2024-01-01', freq='QS')

def generate_economic_series(dates, base_value=100, trend=0.02, volatility=0.05):
    """Generate realistic economic time series"""
    n = len(dates)
    # Trend component
    trend_component = np.cumsum(np.random.normal(trend/12, volatility/4, n))
    # Cyclical component
    cycle = 0.1 * np.sin(2 * np.pi * np.arange(n) / 12) 
    # Random noise
    noise = np.random.normal(0, volatility, n)
    
    return base_value * np.exp(trend_component + cycle + noise)

# Generate monthly indicators
monthly_data = pd.DataFrame({
    'date': monthly_dates,
    'country': 'US',
    'inflation_rate': generate_economic_series(monthly_dates, 2.0, 0.001, 0.02),
    'unemployment_rate': generate_economic_series(monthly_dates, 5.0, -0.001, 0.03),
    'industrial_production': generate_economic_series(monthly_dates, 100, 0.002, 0.04)
})

# Generate quarterly indicators
quarterly_data = pd.DataFrame({
    'date': quarterly_dates,
    'country': 'US', 
    'gdp_growth': generate_economic_series(quarterly_dates, 2.5, 0.0005, 0.015),
    'government_debt': generate_economic_series(quarterly_dates, 80, 0.01, 0.02)
})

print("Monthly data shape:", monthly_data.shape)
print("Quarterly data shape:", quarterly_data.shape)
print("\nMonthly data sample:")
print(monthly_data.head())
print("\nQuarterly data sample:")
print(quarterly_data.head())

Monthly data shape: (49, 5)
Quarterly data shape: (17, 4)

Monthly data sample:
        date country  inflation_rate  unemployment_rate  industrial_production
0 2020-01-01      US        1.935670           4.805587              97.818941
1 2020-02-01      US        2.120364           5.329239             102.432535
2 2020-03-01      US        2.175653           5.432306             105.574865
3 2020-04-01      US        2.209106           5.480127             116.548335
4 2020-05-01      US        2.234182           5.408413             109.315070

Quarterly data sample:
        date country  gdp_growth  government_debt
0 2020-01-01      US    2.545057        80.690558
1 2020-04-01      US    2.657792        82.351275
2 2020-07-01      US    2.765814        85.556844
3 2020-10-01      US    2.827254        87.593422
4 2021-01-01      US    2.744863        87.410966


### Génération de jeux de données de panel

## Détection des délais de publication

### Construction du jeu de données de simulation

In [3]:
# Simulate "existing" data (data up to mid-2023)
existing_monthly = monthly_data[monthly_data['date'] <= '2023-06-01'].copy()
existing_quarterly = quarterly_data[quarterly_data['date'] <= '2023-04-01'].copy()

# Simulate "new" data with some additional observations
new_monthly = monthly_data[monthly_data['date'] <= '2023-08-01'].copy()
new_quarterly = quarterly_data[quarterly_data['date'] <= '2023-07-01'].copy()

print("Existing monthly data ends:", existing_monthly['date'].max())
print("New monthly data ends:", new_monthly['date'].max())
print("Existing quarterly data ends:", existing_quarterly['date'].max())
print("New quarterly data ends:", new_quarterly['date'].max())

Existing monthly data ends: 2023-06-01 00:00:00
New monthly data ends: 2023-08-01 00:00:00
Existing quarterly data ends: 2023-04-01 00:00:00
New quarterly data ends: 2023-07-01 00:00:00


### Détection

In [4]:

new_data = _validate_input_data(data=new_monthly, time_col="date", panel_cols=None)
existing_data = _validate_input_data(data=existing_monthly, time_col="date", panel_cols=None)
# Vérification que les deux DataFrames ont des colonnes en commun
common_columns = list(set(new_data.columns).intersection(set(existing_data.columns)))

# Aucune colonne commune, toutes les observations sont nouvelles
# if not common_data_cols:
#     return self._format_observations_output(new_data, data_cols)

# Restriction aux colonnes communes
new_common_data = new_data[common_columns]
existing_common_data = existing_data[common_columns]

# Alignement des DataFrames sur l'index commun
aligned_new, aligned_existing = new_common_data.align(existing_common_data, fill_value=np.nan)

# Création des masques booléens pour les valeurs nulles
new_isnull = aligned_new.isnull()
existing_isnull = aligned_existing.isnull()

new_isnull.head()

,inflation_rate,country,industrial_production,unemployment_rate
date,,,,
2020-01-01,False,False,False,False
2020-02-01,False,False,False,False
2020-03-01,False,False,False,False
2020-04-01,False,False,False,False
2020-05-01,False,False,False,False


In [5]:
existing_isnull.head()

,inflation_rate,country,industrial_production,unemployment_rate
date,,,,
2020-01-01,False,False,False,False
2020-02-01,False,False,False,False
2020-03-01,False,False,False,False
2020-04-01,False,False,False,False
2020-05-01,False,False,False,False


In [6]:
changes_mask = existing_isnull & ~new_isnull

changes_mask.head()

,inflation_rate,country,industrial_production,unemployment_rate
date,,,,
2020-01-01,False,False,False,False
2020-02-01,False,False,False,False
2020-03-01,False,False,False,False
2020-04-01,False,False,False,False
2020-05-01,False,False,False,False


In [7]:

test = compare_and_detect_delays(
    new_data=new_monthly, 
    existing_data=existing_monthly, 
    download_date="2025-09-05", 
    detection_mode='new_only', 
    reference_point='start', 
    time_col="date", 
    panel_cols=['country'], 
    frequency_detector=None
)

test.head()

column  has_changes
country date                                          
US      2023-07-01         inflation_rate         True
        2023-08-01         inflation_rate         True
        2023-07-01  industrial_production         True
        2023-08-01  industrial_production         True
        2023-07-01      unemployment_rate         True

In [8]:
test.tail()

column  has_changes
country date                                          
US      2023-08-01         inflation_rate         True
        2023-07-01  industrial_production         True
        2023-08-01  industrial_production         True
        2023-07-01      unemployment_rate         True
        2023-08-01      unemployment_rate         True

In [ ]:
from .
download_date="2025-09-05"
reference_point='start'
unit=

In [12]:
("a", "b") + ("c",)

('a', 'b', 'c')

In [ ]:
# Retraitement de la forme
# Ajout de l'indicateur à l'index et suppression de la date
test.set_index("column", drop=True, append=True, inplace=True)
test.reset_index(level=test.index.nlevels-2, drop=False, inplace=True, names=[e for e in range(test.index.nlevels) if e!= test.index.nlevels-2 else "observation_date"])

# Ajout d'informations d'intérêt
# Date de téléchargement
test["download_date"] = download_date
# Fréquence

# Date de début de période

# Date de fin de période



# Point de référence
test["reference_point"] = reference_point
# Délai de publication
# Le délai de publication est toujours arrondi à l'entier supérieur

# Unité

test.head()

date  has_changes
country column                                       
US      inflation_rate        2023-07-01         True
        inflation_rate        2023-08-01         True
        industrial_production 2023-07-01         True
        industrial_production 2023-08-01         True
        unemployment_rate     2023-07-01         True

In [10]:
test.index.nlevels-2

0

In [ ]:
# Fonction de conversion en un dictionnaire associant à chaque entité X colonne, le point de référence et le délai

In [ ]:
list(test.index.names)

['date']

In [16]:
test2 = pd.melt(test, var_name="column", value_name="has_changes", ignore_index=False)
test2.head()

,column,has_changes
date,,
2020-01-01,country,False
2020-02-01,country,False
2020-03-01,country,False
2020-04-01,country,False
2020-05-01,country,False
